# Data handling 

In [1]:
import os
import cv2 as c
import torch as t
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from collections import Counter

comp_train    = "/kaggle/input/datasets/abdelrahmanbakrg/emotion-data/Training_data/Training_data"
comp_test     = "/kaggle/input/datasets/abdelrahmanbakrg/emotion-data/test/test"
ferplus_train = "/kaggle/input/datasets/arnabkumarroy02/ferplus/train"
ferplus_val   = "/kaggle/input/datasets/arnabkumarroy02/ferplus/validation"

device = t.device("cuda" if t.cuda.is_available() else "cpu")

IMG_SIZE      = 128
VALID_CLASSES = {"fear", "happy", "disgust", "sad", "surprise", "angry"}
label_map     = {cls: i for i, cls in enumerate(sorted(VALID_CLASSES))}
idx_to_label  = {v: k for k, v in label_map.items()}
# {'angry':0, 'disgust':1, 'fear':2, 'happy':3, 'sad':4, 'surprise':5}

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

# ── load images ───────────────────────────────────────────────────────────────
def load_images(paths, filter_classes=None):
    images, labels = [], []
    for path in paths:
        for cls_name in os.listdir(path):
            cls_normalized = "surprise" if cls_name == "suprise" else cls_name
            if filter_classes and cls_normalized not in filter_classes:
                continue
            cls_path = os.path.join(path, cls_name)
            if not os.path.isdir(cls_path):
                continue
            for img_file in os.listdir(cls_path):
                if not img_file.endswith(('.jpg', '.png')):
                    continue
                img = c.imread(os.path.join(cls_path, img_file), c.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                images.append(img)
                labels.append(label_map[cls_normalized])
    return images, labels

def load_test_images(path):
    images, filenames = [], []
    for img_file in sorted(os.listdir(path)):  
        if not img_file.endswith('.jpg'):
            continue
        img = c.imread(os.path.join(path, img_file), c.IMREAD_GRAYSCALE)
        if img is None:
            continue
        images.append(img)
        filenames.append(img_file)
    return images, filenames

# ── dataset class ─────────────────────────────────────────────────────────────
class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, t.tensor(label, dtype=t.long)

class TestDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images    = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img

print("Loading training data...")
train_imgs, train_lbls = load_images(
    paths=[comp_train, ferplus_train, ferplus_val],
    filter_classes=VALID_CLASSES
)

print("Loading test data...")
test_imgs, test_filenames = load_test_images(comp_test)

print(f"Total train samples : {len(train_imgs)}")
print(f"Class distribution  : {Counter(train_lbls)}")
print(f"Total test samples  : {len(test_imgs)}")
print(f"Label map           : {label_map}")

class_counts  = Counter(train_lbls)
total_samples = len(train_lbls)
class_weights = t.tensor([
    total_samples / (len(class_counts) * class_counts[i])
    for i in range(len(label_map))
], dtype=t.float32).to(device)

print(f"Class weights       : {class_weights}")

full_dataset = EmotionDataset(train_imgs, train_lbls, transform=train_transform)
val_size     = int(0.15 * len(full_dataset))
train_size   = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                 generator=t.Generator().manual_seed(42))

val_ds.dataset = EmotionDataset(train_imgs, train_lbls, transform=test_transform)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(TestDataset(test_imgs, transform=test_transform), batch_size=32)

print(f"\nTrain batches : {len(train_dl)}")
print(f"Val batches   : {len(val_dl)}")
print(f"Test batches  : {len(test_dl)}")

Loading training data...
Loading test data...
Total train samples : 87237
Class distribution  : Counter({3: 18690, 5: 14032, 4: 14015, 2: 13500, 0: 13500, 1: 13500})
Total test samples  : 7116
Label map           : {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4, 'surprise': 5}
Class weights       : tensor([1.0770, 1.0770, 1.0770, 0.7779, 1.0374, 1.0362], device='cuda:0')

Train batches : 2318
Val batches   : 409
Test batches  : 223


# Model

In [2]:
class VGGNet(nn.Module):
    def __init__(self, num_classes=6):
        super(VGGNet, self).__init__()

        self.features = nn.Sequential(
            # Block 1: (1, 128, 128) → (64, 64, 64)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Block 2: (64, 64, 64) → (128, 32, 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Block 3: (128, 32, 32) → (256, 16, 16)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Block 4: (256, 16, 16) → (512, 8, 8)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Block 5: (512, 8, 8) → (512, 4, 4)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                          # 512 * 4 * 4 = 8192
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = VGGNet(num_classes=6).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 23,639,494


# Training

In [3]:
import pandas as pd
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

model = t.compile(model)

# ── loss + optimizer ──────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-7)

# ── evaluate ──────────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with t.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs    = model(imgs)
            total_loss += criterion(outputs, lbls).item()
            correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
            total      += lbls.size(0)
    return total_loss / len(loader), 100 * correct / total

# ── overfitting check ─────────────────────────────────────────────────────────
def check_overfitting(train_acc, val_acc, train_loss, val_loss, gap_threshold=10.0):
    gap = train_acc - val_acc
    print(f"  ↳ Acc Gap (train - val): {gap:.2f}%", end="  ")
    if gap > gap_threshold:
        print(f"⚠️  Overfitting! gap > {gap_threshold}%")
    elif val_loss > train_loss * 1.3:
        print("⚠️  Overfitting! val_loss >> train_loss")
    else:
        print("✅ OK")

# ── training loop ─────────────────────────────────────────────────────────────
EPOCHS           = 40
PATIENCE         = 10
best_val_acc     = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
        total      += lbls.size(0)

    scheduler.step()
    train_loss_avg    = total_loss / len(train_dl)
    train_acc         = 100 * correct / total
    val_loss, val_acc = evaluate(model, val_dl)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}]  "
          f"Train Loss: {train_loss_avg:.4f}  Train Acc: {train_acc:.2f}%  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    check_overfitting(train_acc, val_acc, train_loss_avg, val_loss)

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        t.save(model.state_dict(), "best_vgg.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop  |  best val acc: {best_val_acc:.2f}%")
            break

print(f"\nBest Val Acc: {best_val_acc:.2f}%")

W0519 22:47:13.261000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch [01/40]  Train Loss: 1.7599  Train Acc: 21.81%  Val Loss: 1.5189  Val Acc: 41.09%
  ↳ Acc Gap (train - val): -19.28%  ✅ OK
Epoch [02/40]  Train Loss: 1.4352  Train Acc: 47.99%  Val Loss: 1.2006  Val Acc: 61.48%
  ↳ Acc Gap (train - val): -13.50%  ✅ OK
Epoch [03/40]  Train Loss: 1.2219  Train Acc: 62.14%  Val Loss: 1.0338  Val Acc: 71.15%
  ↳ Acc Gap (train - val): -9.01%  ✅ OK
Epoch [04/40]  Train Loss: 1.1091  Train Acc: 68.46%  Val Loss: 0.9667  Val Acc: 75.32%
  ↳ Acc Gap (train - val): -6.85%  ✅ OK
Epoch [05/40]  Train Loss: 1.0473  Train Acc: 71.58%  Val Loss: 0.9427  Val Acc: 76.34%
  ↳ Acc Gap (train - val): -4.76%  ✅ OK
Epoch [06/40]  Train Loss: 0.9971  Train Acc: 74.20%  Val Loss: 0.8854  Val Acc: 79.04%
  ↳ Acc Gap (train - val): -4.84%  ✅ OK
Epoch [07/40]  Train Loss: 0.9624  Train Acc: 76.06%  Val Loss: 0.8616  Val Acc: 80.46%
  ↳ Acc Gap (train - val): -4.40%  ✅ OK
Epoch [08/40]  Train Loss: 0.9348  Train Acc: 77.40%  Val Loss: 0.8264  Val Acc: 81.85%
  ↳ Acc Gap (t

# submission 

In [4]:
import re
import unicodedata

def normalize(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def save_submission(prediction_rows, output_dir):
    rows = []
    for name, y_predict in prediction_rows:
        match = re.search(r'([^\\]+)(?=\.\w+$)', name)
        if match:
            raw_name  = match.group(1)
            norm_name = normalize(raw_name)
            match2    = re.search(r'[^/]+$', norm_name)
            if match2:
                rows.append({"ID": match2.group(), "Label": y_predict})
            else:
                raise ValueError(f"No match for: {raw_name}")
    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, 'submission.csv')
    df.to_csv(path, index=False)
    print(f"Updated {len(df)} rows in submission.")
    return df, path

model.load_state_dict(t.load("best_vgg.pth"))

tta_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

TTA_RUNS  = 5
model.eval()
all_preds = []

with t.no_grad():
    for imgs in test_dl:
        imgs  = imgs.to(device)
        probs = t.zeros(imgs.size(0), 6).to(device)
        for _ in range(TTA_RUNS):
            probs += t.softmax(model(imgs), dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().tolist())

prediction_rows = list(zip(test_filenames, [idx_to_label[p] for p in all_preds]))
df, path = save_submission(prediction_rows, output_dir="/kaggle/working")
print(df.head(10))
print(f"\nSaved submission.csv → {len(df)} rows")

Updated 7116 rows in submission.
            ID  Label
0     test (1)  angry
1    test (10)    sad
2   test (100)    sad
3  test (1000)  happy
4  test (1001)  angry
5  test (1002)  angry
6  test (1003)  happy
7  test (1004)    sad
8  test (1005)    sad
9  test (1006)    sad

Saved submission.csv → 7116 rows
